# 04 — Feature Engineering (Gold)

Prepara a base final para baseline e modelo adaptativo: remove leakage, cria features derivadas, seleciona colunas e persiste a camada Gold.


In [ ]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

silver_path = Path("../data/silver/bank_marketing_silver.csv")
df = pd.read_csv(silver_path, sep=";")

print(f"Silver carregada: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()


## 1. Remoção de leakage e colunas pouco úteis


In [ ]:
drop_cols = [c for c in ["duration", "default"] if c in df.columns]
df = df.drop(columns=drop_cols)
print(f"Colunas removidas: {drop_cols}")


## 2. Features derivadas


In [ ]:
df["never_contacted"] = (df["pdays"] == 999).astype(int)

df["previous_success"] = (df["poutcome"].astype(str).str.lower() == "success").astype(int)

df["campaign_bucket"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 3, float("inf")],
    labels=["first_contact", "few_contacts", "many_contacts"],
)

df[["pdays", "never_contacted", "poutcome", "previous_success", "campaign", "campaign_bucket"]].head(10)


## 3. Target binário e seleção final de colunas


In [ ]:
df["y"] = (df["y"].astype(str).str.lower() == "yes").astype(int)

feature_cols = [
    "age",
    "job",
    "marital",
    "education",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
    "never_contacted",
    "previous_success",
    "campaign_bucket",
]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"Colunas esperadas ausentes na Gold: {missing}")

gold_df = df[feature_cols + ["y"]].copy()
print(f"Gold shape: {gold_df.shape}")
print(f"Taxa de conversão: {gold_df['y'].mean():.4f}")
gold_df.head()


## 4. Split train/test (holdout estratificado)

O split é salvo junto da Gold para reprodutibilidade do baseline/bandit (S2).


In [ ]:
train_df, test_df = train_test_split(
    gold_df,
    test_size=0.2,
    random_state=42,
    stratify=gold_df["y"],
)

print(f"Train: {train_df.shape} | conversão={train_df['y'].mean():.4f}")
print(f"Test:  {test_df.shape} | conversão={test_df['y'].mean():.4f}")


## 5. Persistência da camada Gold + metadata


In [ ]:
output_dir = Path("../data/gold")
output_dir.mkdir(parents=True, exist_ok=True)

gold_path = output_dir / "bank_marketing_gold.csv"
train_path = output_dir / "bank_marketing_gold_train.csv"
test_path = output_dir / "bank_marketing_gold_test.csv"

gold_df.to_csv(gold_path, index=False, sep=";")
train_df.to_csv(train_path, index=False, sep=";")
test_df.to_csv(test_path, index=False, sep=";")

metadata = {
    "dataset_name": "Bank Marketing",
    "layer": "gold",
    "gold_version": "1.0",
    "created_at": datetime.now().strftime("%Y-%m-%d"),
    "source_silver": str(silver_path),
    "rows": int(len(gold_df)),
    "columns": list(gold_df.columns),
    "target": "y",
    "target_encoding": {"no": 0, "yes": 1},
    "dropped_columns": drop_cols,
    "engineered_features": [
        "never_contacted",
        "previous_success",
        "campaign_bucket",
    ],
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "test_size": 0.2,
    "random_state": 42,
    "conversion_rate": float(gold_df["y"].mean()),
}

meta_path = output_dir / "metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print(f"Salvo: {gold_path}")
print(f"Salvo: {train_path}")
print(f"Salvo: {test_path}")
print(f"Salvo: {meta_path}")
metadata
